# Preprocessing & First Model

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import min, max, xxhash64, col, when
from functools import reduce
import glob
import matplotlib.pyplot as plt
import pandas as pd
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.feature import StringIndexer, OneHotEncoder
from IPython.display import display
from pyspark.ml.feature import Imputer
from pyspark.ml.classification import RandomForestClassifier, RandomForestClassificationModel
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.mllib.evaluation import MulticlassMetrics
from pyspark.storagelevel import StorageLevel
from pyspark.ml.classification import GBTClassifier

import os
job_scratch = os.environ["TMPDIR"]  # /scratch/lcheng7/job_#
spark_tmp = os.path.join(job_scratch, "spark_tmp")
os.makedirs(spark_tmp, exist_ok=True)

# Change on creation
TOTAL_CORES = 8
TOTAL_MEM_GB = 128
DRIVER_MEM_GB = 12 # Fixed

EXEC_INSTANCES = TOTAL_CORES - 1
EXEC_MEM_GB = int((TOTAL_MEM_GB - DRIVER_MEM_GB) / EXEC_INSTANCES)

spark = (
    SparkSession.builder
    .appName("fhvhv-model")
    .config("spark.driver.memory", f"{DRIVER_MEM_GB}g")
    .config("spark.local.dir", spark_tmp)
    .config("spark.executor.instances", str(EXEC_INSTANCES))
    .config("spark.executor.memory", f"{EXEC_MEM_GB}g")
    .config("spark.sql.shuffle.partitions", str(EXEC_INSTANCES * 3))
    .config("spark.default.parallelism", str(EXEC_INSTANCES * 3))
    .getOrCreate()
)

print("TMPDIR =", job_scratch)
print("spark.local.dir =", spark.sparkContext.getConf().get("spark.local.dir"))
print("parallelism =", spark.sparkContext.defaultParallelism)
print("executors:", EXEC_INSTANCES, "executor_mem_gb:", EXEC_MEM_GB)

Matplotlib created a temporary cache directory at /scratch/chensley1/job_47219188/matplotlib-sw3dqsck because the default path (/home/jovyan/.cache/matplotlib) is not a writable directory; it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


TMPDIR = /scratch/chensley1/job_47219188
spark.local.dir = /scratch/chensley1/job_47219188/spark_tmp
parallelism = 21
executors: 7 executor_mem_gb: 16


In [2]:
print("MASTER =", spark.sparkContext.master)
print("DRIVER MEM =", spark.sparkContext.getConf().get("spark.driver.memory"))
print("EXEC INSTANCES =", spark.sparkContext.getConf().get("spark.executor.instances"))

MASTER = local[*]
DRIVER MEM = 12g
EXEC INSTANCES = 7


In [3]:
df_clean = spark.read.parquet("my_cleaned_data/fhvhv_dedup")
df = df_clean

df = df.drop(
    "originating_base_num",
    "on_scene_datetime",
    "dispatching_base_num",
    "request_datetime"
)

In [4]:
df_small = df.limit(5).toPandas()
df_small

,hvfhs_license_num,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,trip_miles,trip_time,base_passenger_fare,tolls,bcf,sales_tax,congestion_surcharge,airport_fee,tips,driver_pay,shared_request_flag,shared_match_flag,access_a_ride_flag,wav_request_flag,wav_match_flag
0,HV0003,2022-09-16 13:02:34,2022-09-16 14:18:10,3,1,28.72,4536,101.50,28.58,3.98,0.0,0.0,2.5,0.00,78.13,N,N,,N,N
1,HV0003,2021-07-11 12:24:13,2021-07-11 13:06:10,4,1,13.70,2517,48.38,21.00,2.16,0.0,0.0,2.5,11.08,39.95,N,N,,N,N
2,HV0003,2022-03-13 10:09:12,2022-03-13 10:42:26,4,1,14.34,1994,50.64,21.00,2.22,0.0,0.0,2.5,0.00,39.38,N,N,,N,N
3,HV0003,2019-06-27 14:37:12,2019-06-27 15:46:35,4,1,14.41,4163,74.80,21.00,0.00,0.0,0.0,NaN,0.00,77.62,N,N,,N,
4,HV0003,2022-06-12 12:56:31,2022-06-12 13:34:47,4,1,14.42,2296,58.49,21.00,2.46,0.0,0.0,2.5,0.00,42.73,N,N,,N,N


## Imputing

Missing values were concentrated in policy-related fields such as airport_fee and congestion_surcharge.

For this dataset, a missing airport fee logically means the trip was not an airport trip, so it is reasonable to treat missing values as zero.

Similarly, missing congestion surcharge values likely indicate that the surcharge did not apply to that trip.

Rather than dropping millions of rows, we impute these values to preserve data volume and avoid introducing bias.

In [5]:
df = df.fillna({
    "airport_fee": 0,
    "congestion_surcharge": 0
})

df = df.withColumn(
    "wav_match_flag",
    F.when(
        F.col("wav_match_flag").isNull() |
        (F.length(F.trim(F.col("wav_match_flag").cast("string"))) == 0),
        F.lit("Unknown")
    ).otherwise(F.trim(F.col("wav_match_flag").cast("string")))
)

df.groupBy("wav_match_flag").count().show()

+--------------+---------+
|wav_match_flag|    count|
+--------------+---------+
|       Unknown|110657176|
|             N|607638261|
|             Y| 26977884|
+--------------+---------+



In [6]:
null_summary = df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df.columns
])

null_summary.toPandas().T

,0
hvfhs_license_num,0
pickup_datetime,0
dropoff_datetime,0
PULocationID,0
DOLocationID,0
trip_miles,0
trip_time,0
base_passenger_fare,0
tolls,0
bcf,0


## Scaling/Transforming

In [7]:
num_cols = [
    "log_fare",
    "log_miles",
    "log_minutes",
    "tolls",
    "congestion_surcharge",
    "airport_fee",
    "pickup_hour",
    "pickup_dow",
    "pickup_month",
    "is_weekend",
    "is_night",
    "has_airport_fee",
    "has_congestion_fee",
    "total_surcharges",
    "fare_per_mile"
]

# target (tip)
df1 = df.withColumn("tip_binary", F.when(F.col("tips").cast("double") > 0, 1).otherwise(0))

# numeric sanity casts first
df1 = (
    df1
    .withColumn("base_passenger_fare", F.col("base_passenger_fare").cast("double"))
    .withColumn("trip_miles", F.col("trip_miles").cast("double"))
    .withColumn("trip_time", F.col("trip_time").cast("double"))
    .withColumn("tolls", F.col("tolls").cast("double"))
    .withColumn("bcf", F.col("bcf").cast("double"))
    .withColumn("sales_tax", F.col("sales_tax").cast("double"))
    .withColumn("congestion_surcharge", F.col("congestion_surcharge").cast("double"))
    .withColumn("airport_fee", F.col("airport_fee").cast("double"))
    .withColumn("tips", F.col("tips").cast("double"))
)

# time features
df1 = (
    df1
    .withColumn("trip_minutes", F.col("trip_time") / F.lit(60.0))
    .withColumn("pickup_hour", F.hour("pickup_datetime").cast("double"))
    .withColumn("pickup_dow", F.dayofweek("pickup_datetime").cast("double"))
    .withColumn("pickup_month", F.month("pickup_datetime").cast("double"))
    .withColumn("log_fare", F.when(F.col("base_passenger_fare") >= 0, F.log1p("base_passenger_fare")).otherwise(None))
    .withColumn("log_miles", F.when(F.col("trip_miles") >= 0, F.log1p("trip_miles")).otherwise(None))
    .withColumn("log_minutes", F.when(F.col("trip_minutes") >= 0, F.log1p("trip_minutes")).otherwise(None))
)

# flags
df1 = (
    df1
    .withColumn("is_weekend", F.when(F.dayofweek("pickup_datetime").isin([1, 7]), 1.0).otherwise(0.0))
    .withColumn("is_night", F.when(F.hour("pickup_datetime").isin(list(range(0,6)) + list(range(20,24))), 1.0).otherwise(0.0))
    .withColumn("has_airport_fee", F.when(F.col("airport_fee") > 0, 1.0).otherwise(0.0))
    .withColumn("has_congestion_fee", F.when(F.col("congestion_surcharge") > 0, 1.0).otherwise(0.0))
)

df1 = (
    df1
    .withColumn(
        "total_surcharges",
        F.coalesce("tolls", F.lit(0.0)) +
        F.coalesce("bcf", F.lit(0.0)) +
        F.coalesce("sales_tax", F.lit(0.0)) +
        F.coalesce("congestion_surcharge", F.lit(0.0)) +
        F.coalesce("airport_fee", F.lit(0.0))
    )
)

df1 = (
    df1
    .withColumn(
        "fare_per_mile",
        F.when(F.col("trip_miles") > 0, F.col("base_passenger_fare") / F.col("trip_miles")).otherwise(None)
    )
)

df1.select(
    "tip_binary", "log_fare", "log_miles", "log_minutes", "pickup_hour", 
    "pickup_dow", "pickup_month","is_weekend","is_night",
    "has_airport_fee","has_congestion_fee",
    "total_surcharges", "fare_per_mile"
).limit(5).toPandas()

,tip_binary,log_fare,log_miles,log_minutes,pickup_hour,pickup_dow,pickup_month,is_weekend,is_night,has_airport_fee,has_congestion_fee,total_surcharges,fare_per_mile
0,0,4.629863,3.391820,4.338597,13.0,6.0,9.0,0.0,0.0,1.0,0.0,35.06,3.534123
1,1,3.899545,2.687847,3.760037,12.0,1.0,7.0,1.0,0.0,1.0,0.0,25.66,3.531387
2,0,3.944297,2.730464,3.533200,10.0,1.0,3.0,1.0,0.0,1.0,0.0,25.72,3.531381
3,0,4.328098,2.735017,4.253956,14.0,5.0,6.0,0.0,0.0,0.0,0.0,21.00,5.190840
4,0,4.085808,2.735665,3.670376,12.0,1.0,6.0,1.0,0.0,1.0,0.0,25.96,4.056172


In [9]:
# 1. Convert NaNs to Nulls without changing column names
df_num = df1.replace(float('nan'), None, subset=num_cols)

# 2. Imputation
imputer = Imputer(
    inputCols=num_cols,
    outputCols=[f"{c}_imp" for c in num_cols],
    strategy="median"
)
imputer_model = imputer.fit(df_num)
df_imp = imputer_model.transform(df_num)

# 3. Assemble
assembler_num = VectorAssembler(
    inputCols=[f"{c}_imp" for c in num_cols],
    outputCol="num_features",
    handleInvalid="error"
)
df2 = assembler_num.transform(df_imp)

# 4. Scale: Standardize the features
scaler = StandardScaler(
    inputCol="num_features", 
    outputCol="num_scaled", 
    withMean=True, 
    withStd=True
)
scaler_model = scaler.fit(df2)
df3 = scaler_model.transform(df2)

# 5. View
df3.select("num_features", "num_scaled", "tip_binary").limit(5).toPandas()

,num_features,num_scaled,tip_binary
0,"[4.629862798578463, 3.391820219849558, 4.33859...","[2.7134588173097303, 2.775032795508784, 2.5435...",0
1,"[3.8995454839170334, 2.6878474937846906, 3.760...","[1.6223390398921074, 1.7474890769302862, 1.593...",1
2,"[3.944296565978442, 2.73046379593911, 3.533199...","[1.6891987318613149, 1.8096933527018764, 1.221...",0
3,"[4.328098292648326, 2.7350166493320245, 4.2539...","[2.2626120224536153, 1.816338860023799, 2.4045...",0
4,"[4.085808231199814, 2.7356653681351832, 3.6703...","[1.9006221413280922, 1.8172857531442694, 1.446...",0


In [10]:
print("Rows:", df3.count())
df3.groupBy("tip_binary").count().show()

Rows: 745273321
+----------+---------+
|tip_binary|    count|
+----------+---------+
|         1|116837270|
|         0|628436051|
+----------+---------+



## Encoding

In [11]:
# Categorical

cat_cols = [
    "hvfhs_license_num",
    "shared_request_flag",
    "shared_match_flag",
    "access_a_ride_flag",
    "wav_request_flag",
    "wav_match_flag"
]

df5 = df3

for c in cat_cols:
    df5 = df5.withColumn(c, F.trim(F.col(c).cast("string")))
    df5 = df5.withColumn(
        c,
        F.when(F.col(c).isNull() | (F.length(F.col(c)) == 0), F.lit("Unknown")).otherwise(F.col(c))
    )

for c in cat_cols:
    print(c)
    pdf = df5.groupBy(c).count().orderBy(F.desc("count")).limit(10).toPandas()
    display(pdf)

hvfhs_license_num


,hvfhs_license_num,count
0,HV0003,533991717
1,HV0005,191007781
2,HV0004,13884889
3,HV0002,6388934


shared_request_flag


,shared_request_flag,count
0,N,683038777
1,Y,62234543
2,Unknown,1


shared_match_flag


...

,access_a_ride_flag,count
0,Unknown,479489134
1,N,265784187


wav_request_flag


,wav_request_flag,count
0,N,744488730
1,Y,784590
2,Unknown,1


wav_match_flag


,wav_match_flag,count
0,N,607638261
1,Unknown,110657176
2,Y,26977884


In [12]:
# StringIndex + OneHot Encode

indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in cat_cols
]

encoder = OneHotEncoder(
    inputCols=[f"{c}_idx" for c in cat_cols],
    outputCols=[f"{c}_ohe" for c in cat_cols],
    handleInvalid="keep"
)

df_idx = df5
for idx in indexers:
    df_idx = idx.fit(df_idx).transform(df_idx)

df_enc = encoder.fit(df_idx).transform(df_idx)

In [13]:
# Check new dataset
df_enc.select(
    cat_cols +
    [f"{c}_idx" for c in cat_cols] +
    [f"{c}_ohe" for c in cat_cols]
).limit(3).toPandas().T

,0,1,2
hvfhs_license_num,HV0003,HV0003,HV0003
shared_request_flag,N,N,N
shared_match_flag,N,N,N
access_a_ride_flag,Unknown,Unknown,Unknown
wav_request_flag,N,N,N
wav_match_flag,N,N,N
hvfhs_license_num_idx,0.0,0.0,0.0
shared_request_flag_idx,0.0,0.0,0.0
shared_match_flag_idx,0.0,0.0,0.0
access_a_ride_flag_idx,0.0,0.0,0.0


## Feature Engineering

In [14]:
assembler_all = VectorAssembler(
    inputCols=["num_scaled"] + [f"{c}_ohe" for c in cat_cols],
    outputCol="features",
    handleInvalid="keep"
)

df_final = assembler_all.transform(df_enc).select(
    F.col("tip_binary").cast("int").alias("label"),
    "features",
    "pickup_datetime",
    "PULocationID"
)

df_final.limit(5).toPandas()

,label,features,pickup_datetime,PULocationID
0,0,"(2.7134588173097303, 2.775032795508784, 2.5435...",2022-09-16 13:02:34,3
1,1,"(1.6223390398921074, 1.7474890769302862, 1.593...",2021-07-11 12:24:13,4
2,0,"(1.6891987318613149, 1.8096933527018764, 1.221...",2022-03-13 10:09:12,4
3,0,"(2.2626120224536153, 1.816338860023799, 2.4045...",2019-06-27 14:37:12,4
4,0,"(1.9006221413280922, 1.8172857531442694, 1.446...",2022-06-12 12:56:31,4


In [15]:
df_final.write.mode("overwrite").parquet("processed_data/vectorized_features")

# Modeling

In [3]:
df_final = spark.read.parquet("processed_data/vectorized_features")

In [4]:
df_model = df_final.select("label", "features")

In [6]:
# Verify class imbalance
df_model = df_final.select("label", "features")
df_model.groupBy("label").count().show()

+-----+---------+
|label|    count|
+-----+---------+
|    0|628436051|
|    1|116837270|
+-----+---------+



In [5]:
# Stratified 30% subsample
df_model = df_model.sampleBy("label", fractions={0: 0.3, 1: 0.3}, seed=42)

In [11]:
# Verify class imbalance of subsample
df_model.groupBy("label").count().show()

+-----+---------+
|label|    count|
+-----+---------+
|    0|188529418|
|    1| 35043484|
+-----+---------+



In [6]:
# Calculate balancing ratio
total = 628436051 + 116837270
pos   = 116837270
balancing_ratio = (total - pos) / pos  # ~5.38

# Add class weight column
df_model = df_model.withColumn(
    "class_weight",
    when(col("label") == 1, balancing_ratio).otherwise(1.0)
)

# Split
train_df, test_df = df_model.randomSplit([0.8, 0.2], seed=42)

In [10]:
print("Train size:", train_df.count())
print("Test size:", test_df.count())
train_df.groupBy("label").count().show()
test_df.groupBy("label").count().show()

Train size: 596209516
Test size: 149063805
+-----+---------+
|label|    count|
+-----+---------+
|    0|502737362|
|    1| 93472154|
+-----+---------+

+-----+---------+
|label|    count|
+-----+---------+
|    0|125698689|
|    1| 23365116|
+-----+---------+



## Random Forest Classifier

In [13]:
rf = RandomForestClassifier(
    labelCol="label",
    featuresCol="features",
    weightCol="class_weight",
    numTrees=20,
    maxDepth=10,
    maxBins=16,
    subsamplingRate=0.7,
    featureSubsetStrategy="sqrt",
    seed=42
)

rf_model1 = rf.fit(train_df)

In [14]:
rf_model1.write().overwrite().save("models/rf_30_1")

In [7]:
rf_model1 = RandomForestClassificationModel.load("models/rf_30_1")

In [7]:
rf = RandomForestClassifier(
    labelCol="label",
    featuresCol="features",
    weightCol="class_weight",
    numTrees=10,
    maxDepth=15,
    maxBins=12,
    subsamplingRate=0.7,
    featureSubsetStrategy="sqrt",
    seed=42
)

rf_model2 = rf.fit(train_df)

In [9]:
rf_model2 = RandomForestClassificationModel.load("models/rf_30_2")

In [10]:
pred_test1 = rf_model1.transform(test_df)

In [11]:
pred_test2 = rf_model2.transform(test_df)

In [12]:
evaluator_roc = BinaryClassificationEvaluator(
    labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC"
)
evaluator_pr = BinaryClassificationEvaluator(
    labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderPR"
)

roc_auc1 = evaluator_roc.evaluate(pred_test1)
pr_auc1 = evaluator_pr.evaluate(pred_test1)

print("TEST ROC-AUC1:", roc_auc1)
print("TEST PR-AUC1:", pr_auc1)

TEST ROC-AUC1: 0.6443148487021536
TEST PR-AUC1: 0.2425267830850802


In [13]:
# Training vs. Test error comparison for Random Forest model 1:
pred_train1 = rf_model1.transform(train_df)
roc_auc_train1 = evaluator_roc.evaluate(pred_train1)
pr_auc_train1 = evaluator_pr.evaluate(pred_train1)
print("TRAIN ROC-AUC1:", roc_auc_train1)
print("TRAIN PR-AUC1:", pr_auc_train1)
print("Gap (overfit indicator):", roc_auc_train1 - roc_auc1)

TRAIN ROC-AUC1: 0.6442903320066405
TRAIN PR-AUC1: 0.24266136921845116
Gap (overfit indicator): -2.451669551306601e-05


### Fitting Analysis - RF Model 1

| Set   | ROC-AUC | PR-AUC  |
|-------|---------|---------|
| Train | 0.6443  | 0.2427  |
| Test  | 0.6443  | 0.2425  |
| Gap   | ~0.000  | ~0.000  |

The near-zero gap between train and test performance places this model 
firmly in the underfitting region of the bias-variance tradeoff. The model 
is too simple to overfit, but as a result it is also most likely leaving predictive 
signal on the table. Increasing maxDepth, numTrees, and maxBins would 
probably push both train and test AUC higher before overfitting becomes a concern.

In [14]:
evaluator_roc = BinaryClassificationEvaluator(
    labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC"
)
evaluator_pr = BinaryClassificationEvaluator(
    labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderPR"
)

roc_auc2 = evaluator_roc.evaluate(pred_test2)
pr_auc2 = evaluator_pr.evaluate(pred_test2)

print("TEST ROC-AUC2:", roc_auc2)
print("TEST PR-AUC2:", pr_auc2)

TEST ROC-AUC2: 0.6525965184109062
TEST PR-AUC2: 0.2476343049476255


In [15]:
# Training vs. Test error comparison for Random Forest model 2:
pred_train2 = rf_model2.transform(train_df)
roc_auc_train2 = evaluator_roc.evaluate(pred_train2)
pr_auc_train2 = evaluator_pr.evaluate(pred_train2)
print("TRAIN ROC-AUC2:", roc_auc_train2)
print("TRAIN PR-AUC2:", pr_auc_train2)
print("Gap (overfit indicator):", roc_auc_train2 - roc_auc2)

TRAIN ROC-AUC2: 0.6529596184307643
TRAIN PR-AUC2: 0.2481403108256067
Gap (overfit indicator): 0.00036310001985817575


### Fitting Analysis - RF Model 2

| Set   | ROC-AUC | PR-AUC  |
|-------|---------|---------|
| Train | 0.6530  | 0.2481  |
| Test  | 0.6526  | 0.2476  |
| Gap   | 0.0004  | 0.0005  |

Similar to Model 1, the negligible train/test gap places this model in the 
underfitting region. However, the improvement over Model 1 (~0.008 ROC-AUC) 
confirms that increasing the depth of our trees, even after reducing total number 
of trees and number of bins, likely captures more complex feature interactions 
that predict tipping behavior.

In [20]:
### Ground Truth labels alongside model predictions and probabilities for the RF2 model

# Training set examples
pred_train2.filter(col("prediction") == 1).select("label", "prediction", "probability").show(5, truncate=False)
pred_train2.filter(col("prediction") == 0).select("label", "prediction", "probability").show(5, truncate=False)

# Test set examples
pred_test2.filter(col("prediction") == 1).select("label", "prediction", "probability").show(5, truncate=False)
pred_test2.filter(col("prediction") == 0).select("label", "prediction", "probability").show(5, truncate=False)

+-----+----------+----------------------------------------+
|label|prediction|probability                             |
+-----+----------+----------------------------------------+
|0    |1.0       |[0.4942305098942569,0.5057694901057431] |
|0    |1.0       |[0.4812146052389511,0.5187853947610489] |
|0    |1.0       |[0.4971253793320784,0.5028746206679215] |
|0    |1.0       |[0.4221901755728923,0.5778098244271078] |
|0    |1.0       |[0.44217841051664275,0.5578215894833572]|
+-----+----------+----------------------------------------+
only showing top 5 rows

+-----+----------+----------------------------------------+
|label|prediction|probability                             |
+-----+----------+----------------------------------------+
|0    |0.0       |[0.7939841273823262,0.2060158726176739] |
|0    |0.0       |[0.8129415740296159,0.1870584259703842] |
|0    |0.0       |[0.8138403827449687,0.1861596172550313] |
|0    |0.0       |[0.8254561304023543,0.17454386959764578]|
|0    |0.0     

# Preprocessing Summary

## 1. Imputing

Some variables had missing values, mainly airport_fee and congestion_surcharge. Instead of dropping those rows, I filled the missing values with 0.0. If the fee is missing, it likely means the trip did not include that charge, so replacing it with zero makes sense.

For wav_match_flag, missing and empty values were replaced with “Unknown” so that the categorical encoder could still process those rows.

For the rest of the numeric variables (such as log_fare, log_miles, log_minutes, tolls, and the engineered features), I used Spark’s Imputer with the median strategy. The median works better than the mean here because many of the financial variables are skewed.

## 2. Scaling and Transforming

Several numeric variables were transformed before training the model to make their distributions more manageable.

First, log transformations were applied to base_passenger_fare, trip_miles, and trip_minutes using log1p. These variables were heavily right-skewed in the EDA, so the log transform helps reduce the impact of very large values.

After that, all numeric features were standardized using Spark’s StandardScaler (withMean=True, withStd=True). This step ensures that features with different ranges, such as pickup_hour and log_fare, are on a similar scale during model training.

## 3. Encoding

Six categorical variables were encoded for use in the model:

- **hvfhs_license_num** — service provider (Uber, Lyft, etc.)
- **shared_request_flag** — whether a shared ride was requested
- **shared_match_flag** — whether the shared ride was matched
- **access_a_ride_flag** — accessibility ride indicator
- **wav_request_flag** — wheelchair-accessible vehicle request
- **wav_match_flag** — wheelchair-accessible vehicle match

Each variable was first passed through `StringIndexer` to convert string 
labels to numeric indices, then through `OneHotEncoder` to produce sparse 
binary vectors. The `handleInvalid="keep"` parameter was set on both 
transformers to ensure unseen or null categories do not cause runtime 
failures.

## 4. Feature Engineering

Several new features were constructed using Spark SQL functions to capture 
temporal patterns and trip economics that are not directly represented in 
the raw columns:

- **pickup_hour** — hour of day (0-23), captures time-of-day tipping patterns
- **pickup_dow** — day of week (1-7), captures weekday vs. weekend behavior
- **pickup_month** — month of year, captures seasonal trends
- **is_weekend** — binary flag for Saturday/Sunday trips
- **is_night** — binary flag for trips between 8pm and 6am
- **has_airport_fee** — binary flag indicating airport pickup
- **has_congestion_fee** — binary flag indicating congestion zone trip
- **total_surcharges** — sum of all surcharge components (tolls, bcf, 
  sales_tax, congestion_surcharge, airport_fee)
- **fare_per_mile** — base_passenger_fare divided by trip_miles, 
  capturing trip cost efficiency

## 5. Final Feature Assembly

All preprocessed numeric and encoded categorical features were combined 
into a single sparse feature vector using `VectorAssembler`, producing a 
39-dimensional feature vector per row. The final dataset was written to 
Parquet at `processed_data/vectorized_features` with the schema:

- **label** — binary tip indicator (1 if tips > 0, else 0)
- **features** — 39-dimensional sparse vector (15 scaled numeric + 
  24 OHE categorical dimensions)

The full preprocessing pipeline is reproducible: reading from the 
deduplicated Parquet dataset and executing all cells in order will 
reproduce the exact feature matrix used for model training. When 
paired with the EDA notebook an identical deduplicated dataset can
also be reproduced.

# Fitting Analysis

## Where does the model fit in the fitting graph?

Both Random Forest models appear to be underfitting. The training and test scores are almost identical for both configurations (a gap of about -0.00002 for RF1 and 0.00036 for RF2). This means the models generalize well to unseen data, but they are likely too simple to capture stronger patterns in the dataset.

In terms of the bias–variance tradeoff, the models fall on the underfitting side of the curve. Increasing the model complexity (for example by using deeper trees or more trees) could improve performance before overfitting becomes an issue.

### Hyperparameter Comparison

| Model | numTrees | maxDepth | maxBins | Train ROC-AUC | Test ROC-AUC | Train PR-AUC | Test PR-AUC | Gap     |
|-------|----------|----------|---------|---------------|--------------|--------------|-------------|---------|
| RF1   | 20       | 10       | 16      | 0.6443        | 0.6443       | 0.2427       | 0.2425      | ~0.000  |
| RF2   | 10       | 15       | 12      | 0.6530        | 0.6526       | 0.2481       | 0.2476      | 0.0004  |

Increasing maxDepth from 10 to 15 while reducing numTrees from 20 to 10 led to a small improvement in performance. ROC-AUC increased by about 0.008, and PR-AUC improved by about 0.005.

This suggests that tree depth is an important hyperparameter for this dataset. Deeper trees allow the model to capture more complex relationships between features that may influence tipping behavior.

## Which model performs best and why?

RF Model 2 performs slightly better across all metrics. The improvement is small but consistent, with both ROC-AUC and PR-AUC increasing compared to the first model. At the same time, there was no increase in the train/test gap.

This suggests that the deeper trees helped the model learn more complex patterns in the data. Since the training and test scores are still very close, the model likely has not reached its full capacity yet, and further increases in depth or number of trees may still improve performance.

## Next Models for Milestone 4

**Gradient Boosted Trees (GBT)**: GBT is a natural next model to try based on the current results. Unlike Random Forest, which builds trees independently, GBT builds trees sequentially so each new tree focuses on correcting the errors made by the previous ones. This approach may work better for this problem since tipping prediction can be noisy and somewhat imbalanced. We already defined two GBT configurations in the notebook, but they have not yet been successfully run on the 30% stratified sample.

**XGBoost (Distributed)**: Another model we are considering is distributed XGBoost. It includes parameters like scale_pos_weight that help handle class imbalance. XGBoost is also known for stronger regularization and optimization, so it could potentially improve performance compared to Random Forest.

# Conclusion

## What is the conclusion of the first model?

Two Random Forest classifiers were trained on a 30% stratified subsample of the FHVHV ride-share dataset (about 223 million rows) to predict tipping behavior (binary: tip > 0 vs. no tip). Both models achieved a test ROC-AUC between 0.644 and 0.653, which is noticeably better than the 0.50 random baseline we observed when first testing an unweighted model. The training and test scores were very close for both models, indicating that the models generalize well but are likely underfitting and not capturing all useful patterns in the data.

From the hyperparameter comparison, tree depth appears to have the biggest impact on performance. Increasing maxDepth from 10 to 15 (while reducing numTrees from 20 to 10) improved ROC-AUC by about 0.008. This suggests that tipping behavior may depend on more complex combinations of features, such as fare amount, time of day, and trip characteristics, which deeper trees are better able to capture.

The dataset is also imbalanced, with roughly 84% no-tip and 16% tip trips. To address this, we applied inverse frequency class weighting, giving the positive class a weight of about 5.38× so the model pays more attention to tipping cases during training.

## What can be done to possibly improve it?

Several ways could improve the model performance.

**Model capacity:** One option is to continue tuning the model’s hyperparameters. For example, we could increase the number of trees, increase the number of bins, or train on more than the current 30% stratified sample.

**Feature engineering:** The current features mostly describe individual trips (fare, distance, time of day). Additional features based on historical patterns could help, such as average tip rate by pickup zone, hour of day, or service provider (hvfhs_license_num). These types of aggregated features may capture patterns that single-trip features miss.

**Alternative models:** We could also try different models, such as Gradient Boosted Trees (GBT), which may perform better than Random Forest on noisy and imbalanced datasets like tipping prediction.

## How did distributed computing help with this task?

Training a classifier on 745M ride-share records would be very difficult on a single machine. Using Spark on SDSC Expanse allowed us to distribute the work across multiple workers.

Feature engineering and preprocessing ran across multiple cores in parallel, which reduced runtime compared to a single-threaded process.Also, Spark MLlib’s RandomForestClassifier trains trees across workers, where each worker processes its own data partition. This allows the model to train on very large datasets without loading everything into one machine’s memory. Furthermore, operations like dropDuplicates, groupBy, and sampleBy were handled through Spark’s shuffle system across partitions. Even with Spark, these steps were computationally heavy and required several adjustments to avoid memory issues. Spark can spill intermediate data to disk when memory is not enough. Using the scratch directory with spark.local.dir helped us process datasets larger than available RAM during preprocessing.